# Covariance functions

In this notebook we'll see the most common covariance functions and their properties.

In [ ]:
%%capture
!pip install git+https://github.com/italo-goncalves/geoML.git@claude

In [ ]:
import geoml
import geoml.kernels as kr
import geoml.transform as tr

import numpy as np
import matplotlib.pyplot as plt

## 1-dimensional case

First we generate a one-dimenstional grid to represent the spatial input. The covariance matrices will be computed on these coordinates.

Covariance functions consist of a *kernel* and a *transform*, where the kernel gives the overall shape and properties of the resulting random functions.

In the code below, the `Covariance` class represent the an abstract covariance function, which can be materialized on a set of points with the `self_covariance_matrix` method.

In [ ]:
n_points = 501
start, end = 0, 5
x = geoml.data.Grid1D(start=start, n=n_points, end=end)

gauss_cov = kr.Covariance(kr.Gaussian())
exp_cov = kr.Covariance(kr.Exponential())
sph_cov = kr.Covariance(kr.Spherical())

gauss_cov_mat = gauss_cov.self_covariance_matrix(x.coordinates)
exp_cov_mat = exp_cov.self_covariance_matrix(x.coordinates)
sph_cov_mat = sph_cov.self_covariance_matrix(x.coordinates)

fig, ax = plt.subplots(1, 3, figsize=(15, 5))
ax[0].imshow(gauss_cov_mat, extent=(start, end, start, end))
ax[0].set_title('Gaussian')
ax[1].imshow(exp_cov_mat, extent=(start, end, start, end))
ax[1].set_title('Exponential')
ax[2].imshow(sph_cov_mat, extent=(start, end, start, end))
ax[2].set_title('Spherical')
plt.show()

Each covariance function decays differently with distance.

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(15, 5), sharex=True, sharey=True)
ax[0].plot(x.coordinates, gauss_cov_mat[0])
ax[0].set_title('Gaussian')
ax[1].plot(x.coordinates, exp_cov_mat[0])
ax[1].set_title('Exponential')
ax[2].plot(x.coordinates, sph_cov_mat[0])
ax[2].set_title('Spherical')

for a in ax:
    a.set_xlabel('Distance')
    a.set_ylabel('Covariance')
    a.set_xlim(0, 3)
plt.show()

The covariance matrices can be used to generate samples from the corresponding Gaussian process through the Cholesky decomposition.

In [ ]:
# The Gaussian covariance requires the addition of
# a small value to the main diagonal.
jitter = np.eye(n_points) * 1e-6
gauss_chol = np.linalg.cholesky(gauss_cov_mat + jitter)
exp_chol = np.linalg.cholesky(exp_cov_mat)
sph_chol = np.linalg.cholesky(sph_cov_mat)

# Random independent Gaussian samples
rnd = np.random.normal(size=[n_points, 3])

# The spatially correlated values
gauss_samples = np.matmul(gauss_chol, rnd)
exp_samples = np.matmul(exp_chol, rnd)
sph_samples = np.matmul(sph_chol, rnd)

fig, ax = plt.subplots(3, 1, figsize=(15, 15))
ax[0].plot(x.coordinates, gauss_samples)
ax[0].set_title('Gaussian')
ax[1].plot(x.coordinates, exp_samples)
ax[1].set_title('Exponential')
ax[2].plot(x.coordinates, sph_samples)
ax[2].set_title('Spherical')
plt.show()

The *transform* parameter can be used to control the scale of the data. The most simple transform is the `Isotropic`, that divides the coordinates by the *range* `r`.

In [ ]:
cov_1 = kr.Covariance(kr.Exponential(), transform=tr.Isotropic(r=0.25))
cov_2 = kr.Covariance(kr.Exponential(), transform=tr.Isotropic(r=1.0))
cov_3 = kr.Covariance(kr.Exponential(), transform=tr.Isotropic(r=3.0))

cov_mat_1 = cov_1.self_covariance_matrix(x.coordinates)
cov_mat_2 = cov_2.self_covariance_matrix(x.coordinates)
cov_mat_3 = cov_3.self_covariance_matrix(x.coordinates)

chol_1 = np.linalg.cholesky(cov_mat_1 + jitter)
chol_2 = np.linalg.cholesky(cov_mat_2 + jitter)
chol_3 = np.linalg.cholesky(cov_mat_3 + jitter)

rnd = np.random.normal(size=[n_points, 1])
samples_1 = np.matmul(chol_1, rnd)
samples_2 = np.matmul(chol_2, rnd)
samples_3 = np.matmul(chol_3, rnd)

plt.figure(figsize=(15, 5))
plt.plot(x.coordinates, samples_1)
plt.plot(x.coordinates, samples_2)
plt.plot(x.coordinates, samples_3)
plt.legend(['r=0.25', 'r=1.0', 'r=3.0'])
plt.xlabel(r'$x$')
plt.ylabel(r'$f(x)$')
plt.show()

## 2-dimensional case

The covariances defined above also work in two dimensions.

In [ ]:
grid = geoml.data.Grid2D(start=[start, start], end=[end, end], n=[51, 51])

gauss_cov_mat = gauss_cov.self_covariance_matrix(grid.coordinates)
exp_cov_mat = exp_cov.self_covariance_matrix(grid.coordinates)
sph_cov_mat = sph_cov.self_covariance_matrix(grid.coordinates)

jitter = np.eye(grid.n_data) * 1e-6
gauss_chol = np.linalg.cholesky(gauss_cov_mat + jitter)
exp_chol = np.linalg.cholesky(exp_cov_mat)
sph_chol = np.linalg.cholesky(sph_cov_mat)

# Random independent Gaussian samples
rnd = np.random.normal(size=[grid.n_data, 1])

# The spatially correlated values
gauss_samples = np.matmul(gauss_chol, rnd).reshape(grid.grid_size).T
exp_samples = np.matmul(exp_chol, rnd).reshape(grid.grid_size).T
sph_samples = np.matmul(sph_chol, rnd).reshape(grid.grid_size).T

kwargs = dict(extent=(start, end, start, end), origin='lower', vmin=-3, vmax=3)

fig, ax = plt.subplots(1, 3, figsize=(15, 5))
ax[0].imshow(gauss_samples, **kwargs)
ax[0].set_title('Gaussian')
ax[1].imshow(exp_samples, **kwargs)
ax[1].set_title('Exponential')
ax[2].imshow(sph_samples, **kwargs)
ax[2].set_title('Spherical')
plt.show()

In two dimensions we can have *anisotropy*, meaning the random function can fluctuate faster in a given direction than another. This is modelled by an ellipsis, defining the major and minor axes and the azimuth that the major axis is pointing.

In [ ]:
tr1 = tr.Anisotropy2D(maxrange=4, minrange_fct=0.25, azimuth=265)
tr2 = tr.Anisotropy2D(maxrange=1.5, minrange_fct=0.5, azimuth=150)
tr3 = tr.Anisotropy2D(maxrange=5, minrange_fct=0.1, azimuth=15)

cov_1 = kr.Covariance(kr.Spherical(), transform=tr1)
cov_2 = kr.Covariance(kr.Gaussian(), transform=tr2)
cov_3 = kr.Covariance(kr.Gaussian(), transform=tr3)

cov_mat_1 = cov_1.self_covariance_matrix(grid.coordinates)
cov_mat_2 = cov_2.self_covariance_matrix(grid.coordinates)
cov_mat_3 = cov_3.self_covariance_matrix(grid.coordinates)

chol_1 = np.linalg.cholesky(cov_mat_1 + jitter)
chol_2 = np.linalg.cholesky(cov_mat_2 + jitter)
chol_3 = np.linalg.cholesky(cov_mat_3 + jitter)

samples_1 = np.matmul(chol_1, rnd).reshape(grid.grid_size).T
samples_2 = np.matmul(chol_2, rnd).reshape(grid.grid_size).T
samples_3 = np.matmul(chol_3, rnd).reshape(grid.grid_size).T

fig, ax = plt.subplots(1, 3, figsize=(15, 5))
ax[0].imshow(samples_1, **kwargs)
ax[1].imshow(samples_2, **kwargs)
ax[2].imshow(samples_3, **kwargs)
plt.show()